In [24]:
import pandas as pd
from utils.ff_functions import ff_run_regression
from utils.ff_functions import create_coef_table
from utils.ff_functions import summarise_table

# Data Processing
---
We use monthly U.S. equity returns from July 1963 to December 2013.

**Data:** Monthly US equity returns

**Date Range:** July 1963 - December 2013 (Fama and French 2015)

**Frequency:** Monthly

The datasets are:

- Fama–French 5 factors: MKT–RF, SMB, HML, RMW, CMA
- Risk-free rate (RF)
- Three sets of 25 value-weighted portfolios:
  - Size–Book-to-Market (ME × BE/ME)
  - Size–Investment (ME × INV)
  - Size–Operating Profitability (ME × OP)

In [25]:
PATH = '../data/processed'

# Loading processed factor df
ff5 = pd.read_parquet(f'{PATH}/ff5_factors_monthly.parquet')

# Loading processed 25 portfolio dfs
p_SIZE = pd.read_parquet(f'{PATH}/ff_portfolios_25_ME_BEME.parquet')
p_INV = pd.read_parquet(f'{PATH}/ff_portfolios_25_ME_INV.parquet')
p_OP = pd.read_parquet(f'{PATH}/ff_portfolios_25_ME_OP.parquet')

# Filtering date to year range
ff5 = ff5[ff5['Date'].between('1963-07-01', '2013-12-31')].copy()
p_SIZE = p_SIZE[p_SIZE['Date'].between('1963-07-01', '2013-12-31')].copy()
p_INV = p_INV[p_INV['Date'].between('1963-07-01', '2013-12-31')].copy()
p_OP = p_OP[p_OP['Date'].between('1963-07-01', '2013-12-31')].copy()

# Robustness check
assert len(ff5) == len(p_SIZE), 'Observation sizes do not match'
assert len(ff5) == len(p_INV), 'Observation sizes do not match'
assert len(ff5) == len(p_OP), 'Observation sizes do not match'

# Creating list from columns except Date
ff5_cols = [c for c in ff5.columns if c != 'Date']
p_SIZE_cols = [c for c in p_SIZE.columns if c != 'Date']
p_INV_cols = [c for c in p_INV.columns if c != 'Date']
p_OP_cols = [c for c in p_OP.columns if c != 'Date']

# Applying numeric transformations
ff5[ff5_cols] = ff5[ff5_cols].apply(pd.to_numeric, errors='coerce')
p_SIZE[p_SIZE_cols] = p_SIZE[p_SIZE_cols].apply(pd.to_numeric, errors='coerce')
p_INV[p_INV_cols] = p_INV[p_INV_cols].apply(pd.to_numeric, errors='coerce')
p_OP[p_OP_cols] = p_OP[p_OP_cols].apply(pd.to_numeric, errors='coerce')

# Checking for pseudo-missing values
for port in [p_SIZE, p_INV, p_OP]:
    print(port[port.isin([-999, -99.99]).any(axis=1)])

Empty DataFrame
Columns: [Date, SMALL LoBM, ME1 BM2, ME1 BM3, ME1 BM4, SMALL HiBM, ME2 BM1, ME2 BM2, ME2 BM3, ME2 BM4, ME2 BM5, ME3 BM1, ME3 BM2, ME3 BM3, ME3 BM4, ME3 BM5, ME4 BM1, ME4 BM2, ME4 BM3, ME4 BM4, ME4 BM5, BIG LoBM, ME5 BM2, ME5 BM3, ME5 BM4, BIG HiBM]
Index: []

[0 rows x 26 columns]
Empty DataFrame
Columns: [Date, SMALL LoINV, ME1 INV2, ME1 INV3, ME1 INV4, SMALL HiINV, ME2 INV1, ME2 INV2, ME2 INV3, ME2 INV4, ME2 INV5, ME3 INV1, ME3 INV2, ME3 INV3, ME3 INV4, ME3 INV5, ME4 INV1, ME4 INV2, ME4 INV3, ME4 INV4, ME4 INV5, BIG LoINV, ME5 INV2, ME5 INV3, ME5 INV4, BIG HiINV]
Index: []

[0 rows x 26 columns]
Empty DataFrame
Columns: [Date, SMALL LoOP, ME1 OP2, ME1 OP3, ME1 OP4, SMALL HiOP, ME2 OP1, ME2 OP2, ME2 OP3, ME2 OP4, ME2 OP5, ME3 OP1, ME3 OP2, ME3 OP3, ME3 OP4, ME3 OP5, ME4 OP1, ME4 OP2, ME4 OP3, ME4 OP4, ME4 OP5, BIG LoOP, ME5 OP2, ME5 OP3, ME5 OP4, BIG HiOP]
Index: []

[0 rows x 26 columns]


# Merging and Transforming
---
The five factor dataframe is merged into each portfolio set dataframe. We then create excess returns [$R_i - R_f$] by **substracting the risk-free rate from the portfolio returns for each month, row-wise.**

In [26]:
# Factor df
factors_df = ff5[["Date", "Mkt-RF", "SMB", "HML", "RMW", "CMA", "RF"]].copy()


size_cols = [c for c in p_SIZE.columns if c != "Date"]
inv_cols = [c for c in p_INV.columns if c != "Date"]
op_cols = [c for c in p_OP.columns if c != "Date"]

# Indexing by date and saving risk-free rate column as pd series
rf = factors_df.set_index("Date")["RF"]

# Indexing by date for alignment
p_SIZE = p_SIZE.set_index("Date")
p_INV = p_INV.set_index("Date")
p_OP = p_OP.set_index("Date")

# Converting to excess returns
p_SIZE[size_cols] = p_SIZE[size_cols].sub(rf, axis=0)
p_INV[inv_cols] = p_INV[inv_cols].sub(rf, axis=0)
p_OP[op_cols] = p_OP[op_cols].sub(rf, axis=0)

p_SIZE = p_SIZE.reset_index()
p_INV = p_INV.reset_index()
p_OP = p_OP.reset_index()

# CAPM regressions
---
For each of the three sets of 25 portfolios the regression equation is:
$$
\begin{equation}
R_{i,t} - R_{f,t}
=
\alpha_i
+
\beta_{i,M}\left(R_{M,t} - R_{f,t}\right)
+
\varepsilon_{i,t}
\end{equation}
$$

Where:

$$
\begin{aligned}
R_{i,t} &:\ \text{return on portfolio } i \text{ at time } t \\
R_{f,t} &:\ \text{risk-free rate} \\
R_{M,t} - R_{f,t} &:\ \text{market excess return (MKT--RF)} \\
\alpha_i &:\ \text{pricing error (abnormal return)} \\
\varepsilon_{i,t} &:\ \text{regression residual}
\end{aligned}
$$

In [27]:
# Factors that will be used in al regressions
factors_capm = ["Mkt-RF"]

port_frames = {
    "SIZE" : p_SIZE,
    "INV" : p_INV,
    "OP" : p_OP,
}

results = {}
tables = {}
summaries = {}

for idx, frame in port_frames.items():
    results[idx] = ff_run_regression(frame,factors_df=factors_df, factors=factors_capm)
    tables[idx] = create_coef_table(results[idx], factors=factors_capm)
    summaries[idx] = summarise_table(tables[idx])

# Unpacking
size_table, inv_table, op_table = tables["SIZE"], tables["INV"], tables["OP"]
size_summary, inv_summary, op_summary = summaries["SIZE"], summaries["INV"], summaries["OP"]

# Creating table of summaries
capm_summary_df = pd.DataFrame({
    "SIZE": size_summary,
    "INV": inv_summary,
    "OP": op_summary
}).T

display(capm_summary_df)

,mean(|alpha|),% sig alpha (<0.05),mean(R2),n_portfolios
SIZE,0.263111,40.0,0.747964,25.0
INV,0.268510,60.0,0.784413,25.0
OP,0.206605,40.0,0.789842,25.0


## Interpretation
---
### Mean($|\alpha|$)
The mean alpha is economically large at a range of 0.22-0.28. Since our data is in percentage returns, we can interpret the alpha values as a percentage pricing error per month. We then see that CAPM leaves an average **abnormal return** of 0.25% per month unexplained.

### Percentage (%) of Significant Alphas
We can see that for the size and profitability portfolio sets the percentage of significant alphas is ≈45%, while the investment set is at well over 60%. Although not a formal joint significance test, we can already see an indication that the model is mispricing.

### R-Squared
The R-Squareds are large, ranging from 65% to 69% of the variation in excess returns being explained by the market beta. However, the variation being explained is **time-series variation**, not **cross-sectional**. To assess its ability to correctly price an asset we need to look at the intercept, the alpha.

### Conclusions
There are meaningfully large pricing errors across all portfolio sets. Despite high R squared values, there is strong evidence the CAPM model fails to explain cross-sectional variation in returns.

# FF3 regressions
---

For each of the three sets of 25 portfolios the regression equation is:
$$
\begin{equation}
R_{i,t} - R_{f,t}
=
\alpha_i
+
\beta_{i,M}\left(R_{M,t} - R_{f,t}\right)
+
\beta_{i,S}\,\mathrm{SMB}_t
+
\beta_{i,H}\,\mathrm{HML}_t
+
\varepsilon_{i,t}
\end{equation}
$$

Where:

$$
\begin{aligned}
R_{i,t} &:\ \text{return on portfolio } i \text{ at time } t \\
R_{f,t} &:\ \text{risk-free rate} \\
R_{M,t} - R_{f,t} &:\ \text{market excess return (MKT--RF)} \\
\mathrm{SMB}_t &:\ \text{size factor (Small Minus Big)} \\
\mathrm{HML}_t &:\ \text{value factor (High Minus Low)} \\
\alpha_i &:\ \text{pricing error (abnormal return)} \\
\varepsilon_{i,t} &:\ \text{regression residual}
\end{aligned}
$$

In [28]:
# Factors that will be used in al regressions
factors_ff3 = ["Mkt-RF", "SMB", "HML"]

port_frames = {
    "SIZE" : p_SIZE,
    "INV" : p_INV,
    "OP" : p_OP,
}

results = {}
tables = {}
summaries = {}

for idx, frame in port_frames.items():
    results[idx] = ff_run_regression(frame, factors_df=factors_df, factors=factors_ff3)
    tables[idx] = create_coef_table(results[idx], factors=factors_ff3)
    summaries[idx] = summarise_table(tables[idx])

# Unpacking
size_table, inv_table, op_table = tables["SIZE"], tables["INV"], tables["OP"]
size_summary, inv_summary, op_summary = summaries["SIZE"], summaries["INV"], summaries["OP"]

# Creating table of all summaries
ff3_summary_df = pd.DataFrame({
    "SIZE": size_summary,
    "INV": inv_summary,
    "OP": op_summary
}).T

display(ff3_summary_df)

,mean(|alpha|),% sig alpha (<0.05),mean(R2),n_portfolios
SIZE,0.096478,20.0,0.914791,25.0
INV,0.108092,28.0,0.920397,25.0
OP,0.103863,24.0,0.907956,25.0


## Interpretation
---
### Mean($|\alpha|$)
Across the portfolio sets, the mean alpha is much smaller than the CAPM model, at a range of 0.11-0.12. Each month the FF3 model fails to fully price the porfolios at an average pricing error of 0.11%. The mispricing of the model almost halved, a huge improvement from CAPM.

### Percentage (%) of Significant Alphas
The percentage of significant alphas fell to a range of 25%-33%. Although an improvement, it is still economically large. To properly assess the model's validity we will run a formal joint significance (GRS) test in the next section.

### R-Squared
The R Squared increases to ≈0.87. This is a large improvement over CAPM and shows how the extra risk factors improved the model's explanatory power.

### Conclusions
The FF3 model is an improvement over the CAPM in reducing pricing errors, with economically smaller alpha values and a lower proportion of significant alphas. The added risk factors also improve its explanatory power, the model better fits the data. It must be noted that FF3 performs wore on the investment and profitability sets as the risk factors for investment and profitability are missing from the model, they are part of FF5.

# FF5 Regressions
---
We run FF5 regression on every portfolio set. The regression equation is:
$$
\begin{equation}
R_{i,t} - R_{f,t}
=
\alpha_i
+
\beta_{i,M}\left(R_{M,t} - R_{f,t}\right)
+
\beta_{i,S}\,\mathrm{SMB}_t
+
\beta_{i,H}\,\mathrm{HML}_t
+
\beta_{i,R}\, \mathrm{RMW}_t
+
\beta_{i,C}\, \mathrm{CMA}_t
+
\varepsilon_{i,t}
\end{equation}
$$

Where:

$$
\begin{aligned}
R_{i,t} &:\ \text{return on portfolio } i \text{ at time } t \\
R_{f,t} &:\ \text{risk-free rate} \\
R_{M,t} - R_{f,t} &:\ \text{market excess return (MKT--RF)} \\
\mathrm{SMB}_t &:\ \text{size factor (Small Minus Big)} \\
\mathrm{HML}_t &:\ \text{value factor (High Minus Low)} \\
\mathrm{RMW}_t &:\ \text{profitability factor (Robust Minus Weak)} \\
\mathrm{CMA}_t &:\ \text{investment factor (Conservative Minus Aggressive)} \\
\alpha_i &:\ \text{pricing error (abnormal return)} \\
\varepsilon_{i,t} &:\ \text{regression residual}
\end{aligned}
$$

In [29]:
# Factors that will be used in al regressions
factors_ff5 = ["Mkt-RF", "SMB", "HML", "RMW", "CMA"]

port_frames = {
    "SIZE" : p_SIZE,
    "INV" : p_INV,
    "OP" : p_OP,
}

results = {}
tables = {}
summaries = {}

for idx, frame in port_frames.items():
    results[idx] = ff_run_regression(frame, factors_df=factors_df, factors=factors_ff5)
    tables[idx] = create_coef_table(results[idx], factors=factors_ff5)
    summaries[idx] = summarise_table(tables[idx])

# Unpacking
size_table, inv_table, op_table = tables["SIZE"], tables["INV"], tables["OP"]
size_summary, inv_summary, op_summary = summaries["SIZE"], summaries["INV"], summaries["OP"]

# Creating table of all summaries
ff5_summary_df = pd.DataFrame({
    "SIZE": size_summary,
    "INV": inv_summary,
    "OP": op_summary
}).T

display(ff5_summary_df)

,mean(|alpha|),% sig alpha (<0.05),mean(R2),n_portfolios
SIZE,0.088600,32.0,0.918923,25.0
INV,0.082392,12.0,0.931465,25.0
OP,0.071284,8.0,0.929778,25.0


## Interpretation
---
### Mean($|\alpha|$)
Across the portfolio sets, the mean alpha is now economically small at 0.07%-0.09%. Per month, the FF5 model is leaving an average of ≈0.08% abnormal return unexplained.

### Percentage (%) of Significant Alphas
We now see that the profitability portfolio set is approaching the ≈5% significant alphas figure, indicating that the FF5 model prices profitability portfolios well. The value for the investment portfolio set also falls to 12%, a noticeable improvement. The anomaly is the size portfolio set, whose value of significant alphas increased compared to the FF3 model. This is likely due to inflated t-stats due to lowered standard errors.

### R-Squared
The R Squared increases to ≈0.92-0.93. This is a large improvement over CAPM and FF3, showing how the extra risk factors improved the model's explanatory power on excess returns. To reiterate though, since we are testing the model's validity to price a portfolio accurately, the R-Squares is not particularly meaningful.

### Conclusions
FF5 is a sizeable improvement over CAPM and FF3, being particularly strong at pricing the investment and profitability portfolio set. The only limitation is that it is still relatively weak in pricing size portfolios.

---

# Exporting Processed Dataframes

In [30]:
# Exporting excess return portfolio dataframes
p_SIZE.to_parquet("../data/processed/df_SIZE.parquet")
p_INV.to_parquet("../data/processed/df_INV.parquet")
p_OP.to_parquet("../data/processed/df_OP.parquet")

# Exporting factors
factors_df.to_parquet("../data/processed/factors_df.parquet")